# Aspect Labeling (Semi-Supervised)
Penentuan label biner (0/1) untuk masing-masing aspek berdasarkan pencocokan kata kunci (keyword matching). Dataset ini akan menjadi ground truth untuk pelatihan model klasifikasi aspek.

In [ ]:
# Setup environment jika dijalankan di Google Colab / Kaggle
import os
import sys

if 'google.colab' in sys.modules:
    print("Colab detected. Pastikan dataset 'cleaned_reviews.csv' sudah diupload.")

In [ ]:
import pandas as pd
import numpy as np
import os

## 1. Load Data

In [ ]:
dataset_path = '../../preprocessing/cleaned_reviews.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'cleaned_reviews.csv'

df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")

## 2. Definisi Kamus Aspek

In [ ]:
aspect_keywords = {
    'aspek_rasa': ['kopi', 'minuman', 'rasa', 'enak', 'pahit', 'manis', 'es', 'susu', 'roti', 'menu', 'cup', 'kualitas', 'asin', 'gurih', 'cokelat', 'latte', 'matcha', 'boba', 'tawar', 'encer', 'pekat', 'panas', 'dingin'],
    'aspek_harga': ['harga', 'mahal', 'murah', 'promo', 'diskon', 'cashback', 'ovo', 'gopay', 'shopeepay', 'worth', 'saku', 'pas', 'hemat', 'paket', 'pahe', 'mehong'],
    'aspek_pelayanan': ['pelayanan', 'layan', 'kasir', 'barista', 'staff', 'karyawan', 'ramah', 'sopan', 'jutek', 'galak', 'cuek', 'etika', 'respect', 'sapa', 'senyum', 'baik', 'mbak', 'mas'],
    'aspek_kecepatan': ['lama', 'cepat', 'antri', 'nunggu', 'lambat', 'tunggu', 'antrean', 'gercep', 'lelet', 'durasi', 'menit', 'jam', 'kelewat'],
    'aspek_kebersihan': ['bersih', 'kotor', 'toilet', 'tempat', 'nyaman', 'ac', 'meja', 'kursi', 'suasana', 'rapi', 'cozy', 'wfc', 'colokan', 'colok', 'sejuk', 'dingin', 'berisik', 'luas', 'sempit', 'sofa'],
    'aspek_stok': ['habis', 'stok', 'menu', 'sedia', 'kosong', 'varian', 'kehabisan', 'ready', 'sold', 'out'],
    'aspek_aplikasi': ['app', 'aplikasi', 'grab', 'gojek', 'gofood', 'grabfood', 'order', 'pesan', 'sistem', 'pickup', 'kiosk', 'kks', 'down', 'error', 'apk']
}

## 3. Pelabelan Ulasan

In [ ]:
def label_review_aspects(text, keywords):
    if not isinstance(text, str):
        return pd.Series([0] * len(keywords))
    
    text_lower = text.lower()
    labels = {}
    for aspect, kws in keywords.items():
        labels[aspect] = 1 if any(kw in text_lower for kw in kws) else 0
    return pd.Series(labels)

# Eksekusi pelabelan
label_cols = df['text_clean'].apply(lambda x: label_review_aspects(x, aspect_keywords))
df_labeled = pd.concat([df, label_cols], axis=1)

# Identifikasi review umum (tidak masuk ke aspek manapun)
df_labeled['aspek_umum'] = (df_labeled[list(aspect_keywords.keys())].sum(axis=1) == 0).astype(int)
print("Proses pelabelan selesai.")

## 4. Distribusi Label Aspek

In [ ]:
print("=== Sebaran Sampel Per Aspek ===")
stats = {}
for col in list(aspect_keywords.keys()) + ['aspek_umum']:
    count = df_labeled[col].sum()
    percentage = (count / len(df_labeled)) * 100
    stats[col] = int(count)
    print(f"{col}: {count} ulasan ({percentage:.2f}%)")

## 5. Validasi Hasil Pelabelan (Sampel Acak)

In [ ]:
sample_cols = ['ulasan', 'text_clean'] + list(aspect_keywords.keys())
df_labeled[sample_cols].sample(10, random_state=42)

## 6. Simpan Dataset Berlabel

In [ ]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/cleaned_reviews_with_aspect_labels.csv'
df_labeled.to_csv(output_path, index=False)
print(f"Dataset berlabel disimpan di: {output_path}")